# Deep Semantic Search v3 — Full Demo

This notebook demonstrates the current `deep-semantic-search` API:

1. **Image Search** — SigLIP-based image/text retrieval
2. **Text Search** — BGE-M3 dense+sparse hybrid retrieval
3. **Image Clustering** — KMeans or HDBSCAN clustering
4. **Image Captioning** — Florence-2 captions
5. **RAG (Question Answering)** — BGE-M3 retrieval + LiteLLM/Ollama generation
6. **Unified Search** — Cross-modal indexing and retrieval
7. **Duplicate Detection** — Near-duplicate image/text detection

## Setup

Install from source (recommended for this project workspace):

```bash
pip install -e ..
```

In [ ]:
import os
from pathlib import Path

# Paths to demo data
DEMO_DIR = Path(".").resolve()
IMAGE_DIR = DEMO_DIR / "data" / "images"
TEXT_DIR = DEMO_DIR / "data" / "texts"

print(f"Demo directory: {DEMO_DIR}")
print(f"Image directory: {IMAGE_DIR} ({len(list(IMAGE_DIR.iterdir()))} files)")
print(f"Text directory: {TEXT_DIR} ({len(list(TEXT_DIR.iterdir()))} files)")

## 1. Image Search

Load images, build a SigLIP index, and search by text or by image.

In [ ]:
from deep_semantic_search import LoadImageData, ImageIndexer, ImageSearcher

# Load all images from the demo folder
loader = LoadImageData()
image_paths = loader.from_folder([str(IMAGE_DIR)])
print(f"Loaded {len(image_paths)} images")
for p in image_paths:
    print(f"  {os.path.basename(p)}")

In [ ]:
# Build the SigLIP index (auto-skips if already built)
indexer = ImageIndexer(
    image_paths,
    metadata_dir=DEMO_DIR / "metadata" / "siglip_index",
)
indexer.run_index(reindex=True)
print(f"Indexed {len(indexer.image_data)} images")

In [ ]:
# Search by text query
searcher = ImageSearcher(indexer)
results = searcher.search_by_text("a blue sky with water", n=5)

print("\nSearch results for 'a blue sky with water':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Search by text — different query
results = searcher.search_by_text("warm red colors, sunset", n=5)

print("\nSearch results for 'warm red colors, sunset':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Search by image — find images similar to the first one
query_image = image_paths[0]
results = searcher.search_by_image(query_image, n=5)

print(f"\nImages similar to '{os.path.basename(query_image)}':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Visualize similar images (opens matplotlib plots)
searcher.plot_similar_images(query_image, n=4)

## 2. Text Search

Load text documents, create BGE-M3 embeddings, and run hybrid similarity search.

In [ ]:
from deep_semantic_search import LoadTextData, TextEmbedder, TextSearch

# Load text files
text_loader = LoadTextData()
corpus = text_loader.from_folder(str(TEXT_DIR))

print(f"Loaded {len(corpus)} documents:")
for path, text in corpus.items():
    preview = text[:80].replace('\n', ' ')
    print(f"  {os.path.basename(path)}: \"{preview}...\"")

In [ ]:
# Create BGE-M3 embeddings (dense+sparse)
embedder = TextEmbedder(
    metadata_dir=DEMO_DIR / "metadata" / "text_embeddings"
)
embedder.embed(corpus, reindex=True)
print("Embeddings created successfully!")

In [ ]:
# Search for similar documents
search = TextSearch(embedder)

queries = [
    "How does SigLIP work for image understanding?",
    "What is retrieval augmented generation?",
    "Which Python libraries are popular for AI?",
    "How do neural networks learn from data?",
]

for query in queries:
    results = search.find_similar(query, top_n=3, hybrid=True)
    print(f"\nQuery: '{query}'")
    for r in results:
        print(f"  Score: {r['score']:.4f}  File: {os.path.basename(r['path'])}")
        print(f"         \"{r['text'][:100].replace(chr(10), ' ')}...\"")

## 3. Image Clustering

Cluster images into groups using KMeans or HDBSCAN on SigLIP feature vectors.

In [ ]:
from deep_semantic_search import ImageClusterer

# KMeans clustering (set n_clusters=None for HDBSCAN auto mode)
clusterer = ImageClusterer(indexer)
cluster_df = clusterer.cluster(n_clusters=3)

print("Clustering results:")
print(cluster_df[["images_paths", "cluster", "topic"]].to_string(index=False))

In [ ]:
# View images per cluster
for cluster_id in sorted(cluster_df["cluster"].unique()):
    images = clusterer.get_cluster_images(int(cluster_id))
    print(f"\nCluster {cluster_id} ({len(images)} images):")
    for img in images:
        print(f"  {os.path.basename(img)}")

In [ ]:
# Save clusters to organized folders
save_dir = DEMO_DIR / "output" / "clusters"
clusterer.save_clusters(str(save_dir))
print(f"Clusters saved to {save_dir}")

for d in sorted(save_dir.iterdir()):
    files = list(d.iterdir())
    print(f"  {d.name}/  ({len(files)} images)")

In [ ]:
# Plot a cluster
clusterer.plot_cluster(0, n=6)

## 4. Image Captioning

Generate natural language captions for images using Florence-2.

> **Note:** This downloads the Florence-2 model on first run and may take time depending on your hardware/network.

In [ ]:
from deep_semantic_search import ImageCaptioner
import pandas as pd

captioner = ImageCaptioner()

# Caption a subset of images
sample_paths = image_paths[:5]

try:
    captions_df = captioner.caption(sample_paths)
    print("Image Captions:")
    for _, row in captions_df.iterrows():
        print(f"  {os.path.basename(row['image_path'])}: {row['caption']}")
except Exception as e:
    print(f"Captioning unavailable in this environment: {e}")
    captions_df = pd.DataFrame({"image_path": sample_paths, "caption": ["<unavailable>"] * len(sample_paths)})

In [ ]:
# Visualize captioned images
captioner.plot_captioned_images(captions_df, caption_col="caption")

## 4b. Clustering with Captioning

Combine clustering with Florence-2 captioning to auto-generate topic labels.

In [ ]:
# Cluster with captioner for automatic topic labels
# Note: If LiteLLM/Ollama is not available, topics fall back to generic labels.
try:
    clusterer_with_topics = ImageClusterer(indexer)
    cluster_df_captioned = clusterer_with_topics.cluster(n_clusters=3, captioner=captioner)

    print("Clusters with auto-generated topics:")
    for cluster_id in sorted(cluster_df_captioned["cluster"].unique()):
        cluster_data = cluster_df_captioned[cluster_df_captioned["cluster"] == cluster_id]
        topic = cluster_data["topic"].iloc[0]
        count = len(cluster_data)
        print(f"  Cluster {cluster_id} - Topic: '{topic}' ({count} images)")
except Exception as e:
    print(f"Caption-assisted clustering unavailable in this environment: {e}")

## 5. RAG — Question Answering

Answer questions over text documents using BGE-M3 retrieval + LiteLLM/Ollama generation.

> **Note:** Requires Ollama running locally and `deep-semantic-search[llm]` installed.
>
> ```bash
> # Install Ollama: https://ollama.com
> ollama pull gemma4:e4b
> ```

In [ ]:
from deep_semantic_search import RAG

# Get all text content
text_data = list(corpus.values())
print(f"Using {len(text_data)} documents for RAG")

# Ask a question
question = "What is SigLIP and how is it used for image search?"
print(f"\nQuestion: {question}")

try:
    rag = RAG(rerank=False)
    answer = rag.ask(text_data, question, semantic_chunking=True)
    print(f"\nAnswer: {answer}")
except Exception as e:
    print(f"\nRAG unavailable in this environment: {e}")

In [ ]:
# Ask another question
question2 = "What are the steps in a RAG pipeline?"
print(f"Question: {question2}")

try:
    answer2 = rag.ask(text_data, question2, semantic_chunking=True)
    print(f"\nAnswer: {answer2}")
except Exception as e:
    print(f"\nRAG unavailable in this environment: {e}")

In [ ]:
# RAG with custom parameters
question3 = "What Python libraries are useful for machine learning?"
print(f"Question: {question3}")

try:
    answer3 = rag.ask(
        text_data,
        question3,
        chunk_size=500,
        semantic_chunking=False,
    )
    print(f"\nAnswer: {answer3}")
except Exception as e:
    print(f"\nRAG unavailable in this environment: {e}")

## 6. Unified Search

Index images + text in one shared space and run cross-modal queries.

In [ ]:
from deep_semantic_search import UnifiedIndexer, UnifiedSearcher

try:
    unified_indexer = UnifiedIndexer(metadata_dir=DEMO_DIR / "metadata" / "unified")
    unified_indexer.add_images(image_paths)
    unified_indexer.add_texts(list(corpus.values()), labels=list(corpus.keys()))
    unified_indexer.build_index()

    unified_searcher = UnifiedSearcher(unified_indexer)
    unified_results = unified_searcher.search("sunset over ocean", n=5)

    print("Unified search results:")
    for r in unified_results:
        print(f"  {r['rank']}. [{r['type']}] {r['score']:.4f} - {r['source']}")
except Exception as e:
    print(f"Unified search unavailable in this environment: {e}")

## 7. Duplicate Detection

Find near-duplicate images and text entries using embedding similarity thresholds.

In [ ]:
# Image duplicates
img_dupes = searcher.find_duplicates(threshold=0.95)
print(f"Image duplicates found: {len(img_dupes)}")
for p1, p2, sim in img_dupes[:5]:
    print(f"  {sim:.4f} - {os.path.basename(p1)} <-> {os.path.basename(p2)}")

# Text duplicates
text_dupes = search.find_duplicates(threshold=0.95)
print(f"\nText duplicates found: {len(text_dupes)}")
for p1, p2, sim in text_dupes[:5]:
    print(f"  {sim:.4f} - {os.path.basename(p1)} <-> {os.path.basename(p2)}")

## Cleanup

Run this cell to remove generated metadata and output files.

In [ ]:
import shutil

for d in ["metadata", "output"]:
    path = DEMO_DIR / d
    if path.exists():
        shutil.rmtree(path)
        print(f"Removed {path}")

print("Cleanup complete!")